# 18tB — Admissible historical information sets

This stage applies the canonical HKO publication time from 18tA
to the deterministic forecast residual history.

For a current decision \((d,r)\), a historical residual
observation \((u,s)\) is admissible only when

\[
u<d,\qquad
\tau^{\mathrm{HKO},*}(u)
\leq \tau_{d,r},
\qquad
\tau^{\mathrm{weather}}_{u,s}
\leq \tau_{u,s}.
\]

The residual correction target is

\[
R_{u,s}
=
T_u^{\mathrm{HKO}}
-
\widehat T_{u,s}^{\mathrm{det}}.
\]

Both pooled all-rule histories and same-decision-rule histories are
frozen. The output is model-agnostic: it does not choose the final
GP kernel, tree specification, minimum history threshold or
chronological development/holdout split.

No market probability, market price or contract payoff is used to
construct the residual history.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / ".git").exists():
    raise RuntimeError(
        f"Run this notebook from the repository root, not {ROOT}"
    )

UTC = timezone.utc
STEP = "18tB"

RULES = [
    "24h_prior",
    "12h_prior",
    "6h_prior",
    "event_day_open",
]
RULE_ORDER = {
    rule: index
    for index, rule in enumerate(RULES)
}

AVAILABILITY_DIR = (
    ROOT
    / "data/processed/18tA_hko_publication_availability"
)
AVAILABILITY_PATH = (
    AVAILABILITY_DIR
    / "18tA_hko_publication_availability_panel.csv"
)
AVAILABILITY_SUMMARY_PATH = (
    AVAILABILITY_DIR / "18tA_summary.json"
)
AVAILABILITY_MANIFEST_PATH = (
    AVAILABILITY_DIR / "18tA_sha256_manifest.csv"
)

SAMPLE_DIR = (
    ROOT
    / "data/processed/18s_expanded_march_june_canonical_sample"
)
SUPPORT_PATH = (
    SAMPLE_DIR / "18s_expanded_support_audit_panel.csv"
)
WEATHER_PATH = (
    SAMPLE_DIR
    / "18s_expanded_deterministic_weather_panel.csv"
)
SAMPLE_SUMMARY_PATH = (
    SAMPLE_DIR / "18s_expanded_sample_summary.json"
)

OUT_DIR = (
    ROOT
    / "data/processed/18tB_admissible_historical_information_sets"
)
REPORT_DIR = (
    ROOT
    / "reports/18tB_admissible_historical_information_sets"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_INPUTS = [
    AVAILABILITY_PATH,
    AVAILABILITY_SUMMARY_PATH,
    AVAILABILITY_MANIFEST_PATH,
    SUPPORT_PATH,
    WEATHER_PATH,
    SAMPLE_SUMMARY_PATH,
]

for path in REQUIRED_INPUTS:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required input is missing: {path}"
        )

EXPECTED = {
    "settlement_dates": 103,
    "decision_inventory_rows": 412,
    "current_prediction_eligible_rows": 375,
    "residual_observation_rows": 375,
    "support_contract_rows": 4532,
    "weather_contract_rows": 4125,
}

print(f"Repository root: {ROOT}")

Repository root: /Users/edwardlee/Desktop/2026MScWeatherForecastingPolymarket


In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()


def parse_bool(
    series: pd.Series,
    *,
    name: str,
) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    parsed = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "yes": True,
                "no": False,
            }
        )
    )

    if parsed.isna().any():
        bad = series.loc[
            parsed.isna()
        ].drop_duplicates().tolist()
        raise ValueError(
            f"Could not parse Boolean column {name}: {bad}"
        )

    return parsed.astype(bool)


with AVAILABILITY_SUMMARY_PATH.open(
    encoding="utf-8"
) as handle:
    availability_summary = json.load(handle)

with SAMPLE_SUMMARY_PATH.open(
    encoding="utf-8"
) as handle:
    sample_summary = json.load(handle)

if availability_summary.get("verdict") != "PASS":
    raise AssertionError(
        "18tA is not a PASS release."
    )

if sample_summary.get("verdict") != "PASS":
    raise AssertionError(
        "18s is not a PASS release."
    )

if int(
    availability_summary.get("settlement_dates", -1)
) != EXPECTED["settlement_dates"]:
    raise AssertionError(
        "18tA does not contain 103 settlement dates."
    )

if int(
    sample_summary.get(
        "contract_decision_candidates",
        -1,
    )
) != EXPECTED["support_contract_rows"]:
    raise AssertionError(
        "18s support input does not contain 4,532 rows."
    )

if int(
    sample_summary.get("weather_ready_rows", -1)
) != EXPECTED["weather_contract_rows"]:
    raise AssertionError(
        "18s weather input does not contain 4,125 rows."
    )

availability = pd.read_csv(
    AVAILABILITY_PATH,
    low_memory=False,
)
support = pd.read_csv(
    SUPPORT_PATH,
    dtype={"market_id": str},
    low_memory=False,
)
weather = pd.read_csv(
    WEATHER_PATH,
    dtype={"market_id": str},
    low_memory=False,
)

availability["event_date"] = pd.to_datetime(
    availability["event_date"],
    errors="raise",
)
availability[
    "hko_publication_available_hkt"
] = pd.to_datetime(
    availability[
        "hko_publication_available_hkt"
    ],
    utc=True,
    errors="raise",
).dt.tz_convert("Asia/Hong_Kong")
availability[
    "hko_publication_available_utc"
] = pd.to_datetime(
    availability[
        "hko_publication_available_utc"
    ],
    utc=True,
    errors="raise",
)

support["event_date"] = pd.to_datetime(
    support["event_date"],
    errors="raise",
)
support["decision_cutoff_hkt"] = pd.to_datetime(
    support["decision_cutoff_hkt"],
    utc=True,
    errors="raise",
).dt.tz_convert("Asia/Hong_Kong")
support["decision_cutoff_utc"] = pd.to_datetime(
    support["decision_cutoff_utc"],
    utc=True,
    errors="raise",
)
support["weather_path_ready"] = parse_bool(
    support["weather_path_ready"],
    name="support.weather_path_ready",
)

weather["event_date"] = pd.to_datetime(
    weather["event_date"],
    errors="raise",
)
weather["decision_cutoff_utc"] = pd.to_datetime(
    weather["decision_cutoff_utc"],
    utc=True,
    errors="raise",
)
weather[
    "selected_run_initialisation_utc"
] = pd.to_datetime(
    weather[
        "selected_run_initialisation_utc"
    ],
    utc=True,
    errors="raise",
)
weather[
    "selected_run_available_utc"
] = pd.to_datetime(
    weather[
        "selected_run_available_utc"
    ],
    utc=True,
    errors="raise",
)

if len(availability) != 103:
    raise AssertionError(
        f"Availability rows: {len(availability)}"
    )
if len(support) != 4532:
    raise AssertionError(
        f"Support rows: {len(support)}"
    )
if len(weather) != 4125:
    raise AssertionError(
        f"Weather rows: {len(weather)}"
    )

print("Verified 18s and 18tA inputs: PASS")

Verified 18s and 18tA inputs: PASS


In [3]:
decision_group_columns = [
    "event_date",
    "sample_block",
    "decision_rule",
    "decision_rule_order",
    "decision_cutoff_hkt",
    "decision_cutoff_utc",
]

decision_inventory = (
    support.groupby(
        decision_group_columns,
        as_index=False,
    )
    .agg(
        candidate_contract_rows=("market_id", "size"),
        weather_ready_contract_rows=(
            "weather_path_ready",
            "sum",
        ),
    )
    .sort_values(
        ["event_date", "decision_rule_order"]
    )
    .reset_index(drop=True)
)

decision_inventory[
    "current_prediction_eligible"
] = decision_inventory[
    "weather_ready_contract_rows"
].eq(11)
decision_inventory[
    "current_outcome_included_in_information_set"
] = False
decision_inventory[
    "market_information_used_for_residual_history"
] = False
decision_inventory[
    "final_modelling_split"
] = "UNASSIGNED"

if len(decision_inventory) != 412:
    raise AssertionError(
        f"Expected 412 decision rows, "
        f"found {len(decision_inventory)}"
    )

if not decision_inventory[
    "candidate_contract_rows"
].eq(11).all():
    raise AssertionError(
        "A decision inventory row is not an eleven-contract book."
    )

if int(
    decision_inventory[
        "current_prediction_eligible"
    ].sum()
) != 375:
    raise AssertionError(
        "Expected 375 weather-ready current decisions."
    )

if decision_inventory.duplicated(
    ["event_date", "decision_rule"]
).any():
    raise AssertionError(
        "Duplicate current date-rule decisions."
    )

print("Current decision inventory: PASS")
print(
    "Prediction-eligible current decisions: "
    f"{int(decision_inventory['current_prediction_eligible'].sum()):,}"
)

Current decision inventory: PASS
Prediction-eligible current decisions: 375


In [4]:
residual_group_columns = [
    "event_date",
    "sample_block",
    "decision_rule",
    "decision_rule_order",
]

residual_checks = (
    weather.groupby(
        residual_group_columns,
        as_index=False,
    )
    .agg(
        contract_rows=("market_id", "size"),
        forecast_max_nunique=(
            "forecast_daily_max_c",
            "nunique",
        ),
        hko_max_nunique=(
            "hko_daily_max_c",
            "nunique",
        ),
        decision_cutoff_nunique=(
            "decision_cutoff_utc",
            "nunique",
        ),
        run_initialisation_nunique=(
            "selected_run_initialisation_utc",
            "nunique",
        ),
        run_available_nunique=(
            "selected_run_available_utc",
            "nunique",
        ),
    )
)

if not residual_checks["contract_rows"].eq(11).all():
    raise AssertionError(
        "A weather-ready history path is not an eleven-contract book."
    )

for column in [
    "forecast_max_nunique",
    "hko_max_nunique",
    "decision_cutoff_nunique",
    "run_initialisation_nunique",
    "run_available_nunique",
]:
    if not residual_checks[column].eq(1).all():
        raise AssertionError(
            f"Inconsistent path-level field: {column}"
        )

residual_observations = (
    weather.sort_values(
        [
            "event_date",
            "decision_rule_order",
            "market_id",
        ]
    )
    .drop_duplicates(
        ["event_date", "decision_rule"]
    )
    [
        [
            "event_date",
            "sample_block",
            "decision_rule",
            "decision_rule_order",
            "decision_cutoff_utc",
            "selected_run_initialisation_utc",
            "selected_run_available_utc",
            "selected_run_key",
            "forecast_daily_max_c",
            "hko_daily_max_c",
            "forecast_error_c",
            "source_forecast_max_column",
            "source_run_initialisation_column",
            "source_run_available_column",
            "run_metadata_reconstructed",
        ]
    ]
    .copy()
)

residual_observations = residual_observations.merge(
    availability[
        [
            "event_date",
            "availability_source",
            "exact_publication_timestamp_claim",
            "hko_publication_available_hkt",
            "hko_publication_available_utc",
            "fallback_operational_date",
            "fallback_publication_timestamp_hkt",
        ]
    ],
    on="event_date",
    how="left",
    validate="many_to_one",
)

residual_observations[
    "residual_c"
] = (
    residual_observations["hko_daily_max_c"]
    - residual_observations[
        "forecast_daily_max_c"
    ]
)
residual_observations[
    "residual_definition"
] = "HKO daily maximum minus deterministic forecast maximum"
residual_observations[
    "historical_weather_admissible_at_own_decision"
] = (
    residual_observations[
        "selected_run_available_utc"
    ]
    <= residual_observations[
        "decision_cutoff_utc"
    ]
)
residual_observations[
    "market_information_used"
] = False
residual_observations[
    "probability_bridge_used"
] = False

if len(residual_observations) != 375:
    raise AssertionError(
        f"Expected 375 residual observations, "
        f"found {len(residual_observations)}"
    )

if residual_observations[
    [
        "hko_publication_available_utc",
        "forecast_daily_max_c",
        "hko_daily_max_c",
        "residual_c",
    ]
].isna().any().any():
    raise AssertionError(
        "A residual observation lacks a required field."
    )

if not residual_observations[
    "historical_weather_admissible_at_own_decision"
].all():
    raise AssertionError(
        "A historical weather run was unavailable at its own cutoff."
    )

if not np.isclose(
    residual_observations["residual_c"],
    -residual_observations["forecast_error_c"],
    rtol=0.0,
    atol=1e-12,
).all():
    raise AssertionError(
        "Residual sign is inconsistent with the frozen forecast error."
    )

print("Historical residual observation panel: PASS")
print(f"Residual observations: {len(residual_observations):,}")

Historical residual observation panel: PASS
Residual observations: 375


In [5]:
current = (
    decision_inventory.loc[
        decision_inventory[
            "current_prediction_eligible"
        ]
    ]
    .rename(
        columns={
            "event_date": "current_event_date",
            "sample_block": "current_sample_block",
            "decision_rule": "current_decision_rule",
            "decision_rule_order": (
                "current_decision_rule_order"
            ),
            "decision_cutoff_hkt": (
                "current_decision_cutoff_hkt"
            ),
            "decision_cutoff_utc": (
                "current_decision_cutoff_utc"
            ),
        }
    )
    .copy()
)

history = residual_observations.rename(
    columns={
        "event_date": "history_event_date",
        "sample_block": "history_sample_block",
        "decision_rule": "history_decision_rule",
        "decision_rule_order": (
            "history_decision_rule_order"
        ),
        "decision_cutoff_utc": (
            "history_decision_cutoff_utc"
        ),
        "selected_run_initialisation_utc": (
            "history_run_initialisation_utc"
        ),
        "selected_run_available_utc": (
            "history_run_available_utc"
        ),
        "selected_run_key": "history_run_key",
        "forecast_daily_max_c": (
            "history_forecast_daily_max_c"
        ),
        "hko_daily_max_c": (
            "history_hko_daily_max_c"
        ),
        "forecast_error_c": (
            "history_forecast_error_c"
        ),
        "residual_c": "history_residual_c",
        "availability_source": (
            "history_hko_availability_source"
        ),
        "exact_publication_timestamp_claim": (
            "history_exact_publication_timestamp_claim"
        ),
        "hko_publication_available_hkt": (
            "history_hko_available_hkt"
        ),
        "hko_publication_available_utc": (
            "history_hko_available_utc"
        ),
        "fallback_operational_date": (
            "history_fallback_operational_date"
        ),
        "fallback_publication_timestamp_hkt": (
            "history_fallback_publication_timestamp_hkt"
        ),
    }
).copy()

cross = (
    current.assign(_cross_key=1)
    .merge(
        history.assign(_cross_key=1),
        on="_cross_key",
        how="inner",
    )
    .drop(columns="_cross_key")
)

admissible = cross.loc[
    (
        cross["history_event_date"]
        < cross["current_event_date"]
    )
    & (
        cross["history_hko_available_utc"]
        <= cross["current_decision_cutoff_utc"]
    )
    & (
        cross["history_run_available_utc"]
        <= cross["history_decision_cutoff_utc"]
    )
].copy()

admissible["same_decision_rule"] = (
    admissible["current_decision_rule"]
    == admissible["history_decision_rule"]
)
admissible[
    "history_calendar_days_before_current"
] = (
    admissible["current_event_date"]
    - admissible["history_event_date"]
).dt.days
admissible[
    "history_availability_hours_before_current_cutoff"
] = (
    admissible["current_decision_cutoff_utc"]
    - admissible["history_hko_available_utc"]
).dt.total_seconds() / 3600.0
admissible[
    "history_run_lead_hours_before_own_cutoff"
] = (
    admissible["history_decision_cutoff_utc"]
    - admissible["history_run_available_utc"]
).dt.total_seconds() / 3600.0
admissible[
    "current_outcome_used"
] = False
admissible[
    "market_information_used"
] = False
admissible[
    "probability_bridge_used"
] = False
admissible[
    "final_modelling_split"
] = "UNASSIGNED"

admissible = admissible.sort_values(
    [
        "current_event_date",
        "current_decision_rule_order",
        "history_event_date",
        "history_decision_rule_order",
    ]
).reset_index(drop=True)

same_rule = admissible.loc[
    admissible["same_decision_rule"]
].copy()

if admissible.empty:
    raise AssertionError(
        "No admissible historical pairs were constructed."
    )

if same_rule.empty:
    raise AssertionError(
        "No same-rule historical pairs were constructed."
    )

if not (
    admissible["history_event_date"]
    < admissible["current_event_date"]
).all():
    raise AssertionError(
        "A history date is not strictly before its current date."
    )

if not (
    admissible["history_hko_available_utc"]
    <= admissible["current_decision_cutoff_utc"]
).all():
    raise AssertionError(
        "A historical HKO outcome was unavailable at current cutoff."
    )

if not (
    admissible["history_run_available_utc"]
    <= admissible["history_decision_cutoff_utc"]
).all():
    raise AssertionError(
        "A historical forecast was unavailable at its own cutoff."
    )

if not same_rule[
    "same_decision_rule"
].all():
    raise AssertionError(
        "The same-rule pair panel contains a cross-rule row."
    )

print("Admissible historical pair construction: PASS")
print(f"All-rule history pairs: {len(admissible):,}")
print(f"Same-rule history pairs: {len(same_rule):,}")

Admissible historical pair construction: PASS


All-rule history pairs: 66,575
Same-rule history pairs: 16,638


In [6]:
current_key = [
    "current_event_date",
    "current_decision_rule",
]

all_counts = (
    admissible.groupby(
        current_key,
        as_index=False,
    )
    .agg(
        n_admissible_all_rule_residuals=(
            "history_residual_c",
            "size",
        ),
        n_distinct_all_rule_history_dates=(
            "history_event_date",
            "nunique",
        ),
        earliest_all_rule_history_date=(
            "history_event_date",
            "min",
        ),
        latest_all_rule_history_date=(
            "history_event_date",
            "max",
        ),
        latest_all_rule_hko_availability_utc=(
            "history_hko_available_utc",
            "max",
        ),
    )
)

same_counts = (
    same_rule.groupby(
        current_key,
        as_index=False,
    )
    .agg(
        n_admissible_same_rule_residuals=(
            "history_residual_c",
            "size",
        ),
        n_distinct_same_rule_history_dates=(
            "history_event_date",
            "nunique",
        ),
        earliest_same_rule_history_date=(
            "history_event_date",
            "min",
        ),
        latest_same_rule_history_date=(
            "history_event_date",
            "max",
        ),
        latest_same_rule_hko_availability_utc=(
            "history_hko_available_utc",
            "max",
        ),
        mean_same_rule_residual_c=(
            "history_residual_c",
            "mean",
        ),
        sample_sd_same_rule_residual_c=(
            "history_residual_c",
            "std",
        ),
    )
)

history_summary = (
    decision_inventory.rename(
        columns={
            "event_date": "current_event_date",
            "sample_block": "current_sample_block",
            "decision_rule": "current_decision_rule",
            "decision_rule_order": (
                "current_decision_rule_order"
            ),
            "decision_cutoff_hkt": (
                "current_decision_cutoff_hkt"
            ),
            "decision_cutoff_utc": (
                "current_decision_cutoff_utc"
            ),
        }
    )
    .merge(
        all_counts,
        on=current_key,
        how="left",
        validate="one_to_one",
    )
    .merge(
        same_counts,
        on=current_key,
        how="left",
        validate="one_to_one",
    )
)

count_columns = [
    "n_admissible_all_rule_residuals",
    "n_distinct_all_rule_history_dates",
    "n_admissible_same_rule_residuals",
    "n_distinct_same_rule_history_dates",
]
history_summary[count_columns] = (
    history_summary[count_columns]
    .fillna(0)
    .astype(int)
)

for threshold in [1, 8, 14, 16, 20, 30]:
    history_summary[
        f"same_rule_history_at_least_{threshold}"
    ] = (
        history_summary[
            "n_admissible_same_rule_residuals"
        ]
        >= threshold
    )

history_summary[
    "model_fit_eligibility_not_final"
] = (
    history_summary[
        "current_prediction_eligible"
    ]
    & history_summary[
        "same_rule_history_at_least_8"
    ]
)
history_summary[
    "final_modelling_split"
] = "UNASSIGNED"
history_summary[
    "minimum_history_threshold_selected"
] = False

if len(history_summary) != 412:
    raise AssertionError(
        f"Expected 412 information-set summaries, "
        f"found {len(history_summary)}"
    )

eligible_summary = history_summary.loc[
    history_summary[
        "current_prediction_eligible"
    ]
]

if len(eligible_summary) != 375:
    raise AssertionError(
        "Expected 375 prediction-eligible summary rows."
    )

if not (
    eligible_summary[
        "n_admissible_all_rule_residuals"
    ]
    >= eligible_summary[
        "n_admissible_same_rule_residuals"
    ]
).all():
    raise AssertionError(
        "A same-rule history count exceeds its all-rule count."
    )

rule_summary_rows = []

for rule in RULES:
    rule_rows = history_summary.loc[
        history_summary[
            "current_decision_rule"
        ].eq(rule)
    ]
    eligible_rule = rule_rows.loc[
        rule_rows[
            "current_prediction_eligible"
        ]
    ]

    rule_summary_rows.append(
        {
            "decision_rule": rule,
            "decision_rule_order": RULE_ORDER[rule],
            "decision_rows": len(rule_rows),
            "prediction_eligible_rows": len(
                eligible_rule
            ),
            "rows_with_any_same_rule_history": int(
                eligible_rule[
                    "same_rule_history_at_least_1"
                ].sum()
            ),
            "rows_with_at_least_8_same_rule_history": int(
                eligible_rule[
                    "same_rule_history_at_least_8"
                ].sum()
            ),
            "rows_with_at_least_14_same_rule_history": int(
                eligible_rule[
                    "same_rule_history_at_least_14"
                ].sum()
            ),
            "rows_with_at_least_16_same_rule_history": int(
                eligible_rule[
                    "same_rule_history_at_least_16"
                ].sum()
            ),
            "minimum_same_rule_history_count": int(
                eligible_rule[
                    "n_admissible_same_rule_residuals"
                ].min()
            ),
            "median_same_rule_history_count": float(
                eligible_rule[
                    "n_admissible_same_rule_residuals"
                ].median()
            ),
            "maximum_same_rule_history_count": int(
                eligible_rule[
                    "n_admissible_same_rule_residuals"
                ].max()
            ),
            "all_rule_pair_rows": int(
                admissible[
                    "current_decision_rule"
                ].eq(rule).sum()
            ),
            "same_rule_pair_rows": int(
                same_rule[
                    "current_decision_rule"
                ].eq(rule).sum()
            ),
        }
    )

rule_summary = pd.DataFrame(
    rule_summary_rows
).sort_values(
    "decision_rule_order"
).reset_index(drop=True)

print("Information-set support summaries: PASS")
display(rule_summary)

Information-set support summaries: PASS


,decision_rule,decision_rule_order,decision_rows,prediction_eligible_rows,rows_with_any_same_rule_history,rows_with_at_least_8_same_rule_history,rows_with_at_least_14_same_rule_history,rows_with_at_least_16_same_rule_history,minimum_same_rule_history_count,median_same_rule_history_count,maximum_same_rule_history_count,all_rule_pair_rows,same_rule_pair_rows
0,24h_prior,0,103,96,95,87,80,78,0,44.0,91,16573,4326
1,12h_prior,1,103,93,92,80,75,75,0,44.0,88,16539,4044
2,6h_prior,2,103,91,89,80,75,74,0,44.0,89,16544,3952
3,event_day_open,3,103,95,93,81,80,76,0,46.0,93,16919,4316


In [7]:
check_rows = []

def add_check(
    check: str,
    passed: bool,
    detail: str,
) -> None:
    check_rows.append(
        {
            "check": check,
            "passed": bool(passed),
            "detail": detail,
            "blocking": True,
        }
    )

add_check(
    "18tA_input_pass",
    availability_summary.get("verdict") == "PASS",
    str(availability_summary.get("verdict")),
)
add_check(
    "18s_input_pass",
    sample_summary.get("verdict") == "PASS",
    str(sample_summary.get("verdict")),
)
add_check(
    "decision_inventory_rows_412",
    len(decision_inventory) == 412,
    f"rows={len(decision_inventory)}",
)
add_check(
    "prediction_eligible_decisions_375",
    int(
        decision_inventory[
            "current_prediction_eligible"
        ].sum()
    )
    == 375,
    (
        "eligible="
        f"{int(decision_inventory['current_prediction_eligible'].sum())}"
    ),
)
add_check(
    "residual_observations_375",
    len(residual_observations) == 375,
    f"rows={len(residual_observations)}",
)
add_check(
    "history_dates_strictly_prior",
    (
        admissible["history_event_date"]
        < admissible["current_event_date"]
    ).all(),
    "u < d",
)
add_check(
    "hko_outcomes_available_by_current_cutoff",
    (
        admissible["history_hko_available_utc"]
        <= admissible[
            "current_decision_cutoff_utc"
        ]
    ).all(),
    "tau_HKO*(u) <= tau_current",
)
add_check(
    "historical_weather_available_at_own_cutoff",
    (
        admissible["history_run_available_utc"]
        <= admissible[
            "history_decision_cutoff_utc"
        ]
    ).all(),
    "weather run admissibility",
)
add_check(
    "same_rule_subset_valid",
    same_rule["same_decision_rule"].all(),
    f"rows={len(same_rule)}",
)
add_check(
    "no_current_outcome_in_information_set",
    not admissible["current_outcome_used"].any(),
    "current outcome excluded",
)
add_check(
    "no_market_information_used",
    not admissible["market_information_used"].any(),
    "residual history is weather plus HKO only",
)
add_check(
    "no_probability_bridge_used",
    not admissible["probability_bridge_used"].any(),
    "deterministic residual target only",
)
add_check(
    "final_modelling_split_unassigned",
    history_summary[
        "final_modelling_split"
    ].eq("UNASSIGNED").all(),
    "split assignment deferred",
)
add_check(
    "minimum_history_threshold_not_selected",
    not history_summary[
        "minimum_history_threshold_selected"
    ].any(),
    "threshold flags are descriptive only",
)

integrity = pd.DataFrame(check_rows)

if not integrity["passed"].all():
    raise AssertionError(
        "18tB blocking checks failed:\n"
        + integrity.loc[
            ~integrity["passed"]
        ].to_string(index=False)
    )

issues = pd.DataFrame(
    columns=[
        "issue_level",
        "issue_code",
        "current_event_date",
        "current_decision_rule",
        "history_event_date",
        "history_decision_rule",
        "detail",
        "blocking",
    ]
)

print("18tB integrity checks: PASS")
display(integrity)

18tB integrity checks: PASS


,check,passed,detail,blocking
0,18tA_input_pass,True,PASS,True
1,18s_input_pass,True,PASS,True
2,decision_inventory_rows_412,True,rows=412,True
3,prediction_eligible_decisions_375,True,eligible=375,True
4,residual_observations_375,True,rows=375,True
5,history_dates_strictly_prior,True,u < d,True
6,hko_outcomes_available_by_current_cutoff,True,tau_HKO*(u) <= tau_current,True
7,historical_weather_available_at_own_cutoff,True,weather run admissibility,True
8,same_rule_subset_valid,True,rows=16638,True
9,no_current_outcome_in_information_set,True,current outcome excluded,True


In [8]:
output_frames = {
    "decision_inventory": decision_inventory,
    "residual_observations": residual_observations,
    "admissible_all_rules": admissible,
    "admissible_same_rule": same_rule,
    "history_summary": history_summary,
    "rule_summary": rule_summary,
    "integrity": integrity,
    "issues": issues,
}

output_paths = {
    "decision_inventory": (
        OUT_DIR / "18tB_decision_inventory.csv"
    ),
    "residual_observations": (
        OUT_DIR / "18tB_residual_observation_panel.csv"
    ),
    "admissible_all_rules": (
        OUT_DIR
        / "18tB_admissible_history_pairs_all_rules.csv"
    ),
    "admissible_same_rule": (
        OUT_DIR
        / "18tB_admissible_history_pairs_same_rule.csv"
    ),
    "history_summary": (
        OUT_DIR / "18tB_information_set_summary.csv"
    ),
    "rule_summary": (
        OUT_DIR
        / "18tB_information_set_summary_by_rule.csv"
    ),
    "integrity": (
        OUT_DIR / "18tB_integrity_checks.csv"
    ),
    "issues": (
        OUT_DIR / "18tB_issues.csv"
    ),
}

for key, frame in output_frames.items():
    output = frame.copy()

    for column in output.columns:
        if "date" in column.lower():
            if pd.api.types.is_datetime64_any_dtype(
                output[column]
            ):
                output[column] = output[
                    column
                ].dt.strftime("%Y-%m-%d")

        if (
            "timestamp" in column.lower()
            or column.lower().endswith("_utc")
            or column.lower().endswith("_hkt")
            or "initialisation" in column.lower()
            or "initialization" in column.lower()
            or "available" in column.lower()
        ):
            output[column] = output[column].astype(
                "string"
            )

    output.to_csv(
        output_paths[key],
        index=False,
    )

source_inventory = pd.DataFrame(
    [
        {
            "input_role": "18tA_availability_panel",
            "path": str(
                AVAILABILITY_PATH.relative_to(ROOT)
            ),
            "rows": len(availability),
            "sha256": sha256_file(
                AVAILABILITY_PATH
            ),
        },
        {
            "input_role": "18tA_summary",
            "path": str(
                AVAILABILITY_SUMMARY_PATH.relative_to(
                    ROOT
                )
            ),
            "rows": 1,
            "sha256": sha256_file(
                AVAILABILITY_SUMMARY_PATH
            ),
        },
        {
            "input_role": "18s_support_audit",
            "path": str(
                SUPPORT_PATH.relative_to(ROOT)
            ),
            "rows": len(support),
            "sha256": sha256_file(SUPPORT_PATH),
        },
        {
            "input_role": "18s_weather_panel",
            "path": str(
                WEATHER_PATH.relative_to(ROOT)
            ),
            "rows": len(weather),
            "sha256": sha256_file(WEATHER_PATH),
        },
        {
            "input_role": "18s_summary",
            "path": str(
                SAMPLE_SUMMARY_PATH.relative_to(ROOT)
            ),
            "rows": 1,
            "sha256": sha256_file(
                SAMPLE_SUMMARY_PATH
            ),
        },
    ]
)
source_inventory_path = (
    OUT_DIR / "18tB_source_inventory.csv"
)
source_inventory.to_csv(
    source_inventory_path,
    index=False,
)

eligible_history = history_summary.loc[
    history_summary[
        "current_prediction_eligible"
    ]
]

summary = {
    "step": STEP,
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": "PASS",
    "settlement_dates": int(
        availability["event_date"].nunique()
    ),
    "decision_inventory_rows": int(
        len(decision_inventory)
    ),
    "current_prediction_eligible_rows": int(
        decision_inventory[
            "current_prediction_eligible"
        ].sum()
    ),
    "residual_observation_rows": int(
        len(residual_observations)
    ),
    "admissible_all_rule_pair_rows": int(
        len(admissible)
    ),
    "admissible_same_rule_pair_rows": int(
        len(same_rule)
    ),
    "prediction_eligible_rows_with_zero_same_rule_history": int(
        eligible_history[
            "n_admissible_same_rule_residuals"
        ].eq(0).sum()
    ),
    "prediction_eligible_rows_with_at_least_8_same_rule_history": int(
        eligible_history[
            "same_rule_history_at_least_8"
        ].sum()
    ),
    "prediction_eligible_rows_with_at_least_14_same_rule_history": int(
        eligible_history[
            "same_rule_history_at_least_14"
        ].sum()
    ),
    "prediction_eligible_rows_with_at_least_16_same_rule_history": int(
        eligible_history[
            "same_rule_history_at_least_16"
        ].sum()
    ),
    "minimum_same_rule_history_count_on_eligible_rows": int(
        eligible_history[
            "n_admissible_same_rule_residuals"
        ].min()
    ),
    "median_same_rule_history_count_on_eligible_rows": float(
        eligible_history[
            "n_admissible_same_rule_residuals"
        ].median()
    ),
    "maximum_same_rule_history_count_on_eligible_rows": int(
        eligible_history[
            "n_admissible_same_rule_residuals"
        ].max()
    ),
    "history_rule": (
        "history date strictly before current date; "
        "HKO publication available by current cutoff; "
        "historical weather run available by its own cutoff"
    ),
    "residual_definition": (
        "HKO daily maximum minus deterministic forecast maximum"
    ),
    "market_information_used": False,
    "probability_bridge_retained": False,
    "current_outcome_used": False,
    "minimum_history_threshold_selected": False,
    "final_modelling_split_assigned": False,
    "issue_rows": 0,
    "integrity_checks_passed": int(
        integrity["passed"].sum()
    ),
    "integrity_checks_total": int(
        len(integrity)
    ),
    "availability_source_counts": {
        key: int(value)
        for key, value in residual_observations[
            "availability_source"
        ].value_counts().to_dict().items()
    },
    "by_rule": rule_summary.to_dict(
        orient="records"
    ),
}

summary_path = OUT_DIR / "18tB_summary.json"
summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

environment = {
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "revision": "v1",
}
environment_path = (
    OUT_DIR / "18tB_environment.json"
)
environment_path.write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

report_lines = [
    "# 18tB admissible historical information sets",
    "",
    "**PASS**",
    "",
    "## Frozen objects",
    "",
    (
        f"- Current date-rule decisions: "
        f"{len(decision_inventory):,}"
    ),
    (
        "- Current decisions with a deterministic weather path: "
        f"{int(decision_inventory['current_prediction_eligible'].sum()):,}"
    ),
    (
        "- Historical residual observations: "
        f"{len(residual_observations):,}"
    ),
    (
        "- Admissible all-rule history pairs: "
        f"{len(admissible):,}"
    ),
    (
        "- Admissible same-rule history pairs: "
        f"{len(same_rule):,}"
    ),
    "",
    "## Admissibility rule",
    "",
    (
        "A historical date strictly precedes the current "
        "settlement date, its canonical HKO publication time is "
        "no later than the current decision cut-off, and its "
        "weather run was available by its own historical cut-off."
    ),
    "",
    "## Support by current decision rule",
    "",
    (
        "| Rule | Decisions | Prediction eligible | "
        "Any same-rule history | At least 8 | At least 14 | "
        "At least 16 | Median history | Maximum history |"
    ),
    "|---|---:|---:|---:|---:|---:|---:|---:|---:|",
]

for row in rule_summary.itertuples(index=False):
    report_lines.append(
        "| {rule} | {decisions} | {eligible} | {any_history} | "
        "{eight} | {fourteen} | {sixteen} | {median:.1f} | "
        "{maximum} |".format(
            rule=row.decision_rule,
            decisions=int(row.decision_rows),
            eligible=int(row.prediction_eligible_rows),
            any_history=int(
                row.rows_with_any_same_rule_history
            ),
            eight=int(
                row.rows_with_at_least_8_same_rule_history
            ),
            fourteen=int(
                row.rows_with_at_least_14_same_rule_history
            ),
            sixteen=int(
                row.rows_with_at_least_16_same_rule_history
            ),
            median=float(
                row.median_same_rule_history_count
            ),
            maximum=int(
                row.maximum_same_rule_history_count
            ),
        )
    )

report_lines.extend(
    [
        "",
        "## Methodological boundary",
        "",
        (
            "The residual history uses only HKO outcomes and "
            "deterministic weather forecasts. Market prices, "
            "contract probabilities, the archived Gaussian "
            "bridge and the current outcome are excluded."
        ),
        "",
        (
            "Threshold flags are descriptive. The final minimum "
            "history threshold, model specification and "
            "chronological split remain unassigned."
        ),
    ]
)

report_path = (
    REPORT_DIR
    / "18tB_admissible_historical_information_sets_report.md"
)
report_path.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)

manifest_rows = []
for root in [OUT_DIR, REPORT_DIR]:
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.name == "18tB_sha256_manifest.csv":
            continue
        manifest_rows.append(
            {
                "path": str(path.relative_to(ROOT)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

manifest_path = (
    OUT_DIR / "18tB_sha256_manifest.csv"
)
pd.DataFrame(manifest_rows).to_csv(
    manifest_path,
    index=False,
)

print(json.dumps(summary, indent=2))
print("18tB admissible-history release: PASS")

{
  "step": "18tB",
  "generated_at_utc": "2026-07-21T21:17:45.917131+00:00",
  "verdict": "PASS",
  "settlement_dates": 103,
  "decision_inventory_rows": 412,
  "current_prediction_eligible_rows": 375,
  "residual_observation_rows": 375,
  "admissible_all_rule_pair_rows": 66575,
  "admissible_same_rule_pair_rows": 16638,
  "prediction_eligible_rows_with_zero_same_rule_history": 6,
  "prediction_eligible_rows_with_at_least_8_same_rule_history": 328,
  "prediction_eligible_rows_with_at_least_14_same_rule_history": 310,
  "prediction_eligible_rows_with_at_least_16_same_rule_history": 303,
  "minimum_same_rule_history_count_on_eligible_rows": 0,
  "median_same_rule_history_count_on_eligible_rows": 44.0,
  "maximum_same_rule_history_count_on_eligible_rows": 93,
  "history_rule": "history date strictly before current date; HKO publication available by current cutoff; historical weather run available by its own cutoff",
  "residual_definition": "HKO daily maximum minus deterministic forecast